<a href="https://colab.research.google.com/github/RamyHamrouni/LLM-Fine-tuning/blob/main/sft_llama3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

You can now train embedding models 1.8-3.3x faster with 20% less VRAM. [Blog](https://unsloth.ai/docs/new/embedding-finetuning)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

3x faster LLM training with 30% less VRAM and 500K context. [3x faster](https://unsloth.ai/docs/new/3x-faster-training-packing) • [500K Context](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [2]:
%%capture
import os

!pip install pip3-autoremove

!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [58]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",

    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",


    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct", # or choose "unsloth/Llama-3.2-1B-Instruct"
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [59]:
from unsloth import FastLanguageModel

# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

# Define your test message
messages = [
    {"role": "user", "content": "What do you know about model context protocol"},
]

# Apply chat template and tokenize
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be True for inference
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
)

# Decode only the new tokens (skip the input prompt)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Model response:", response)

Model response: The Model Context Protocol (MCP) is a protocol used for communication between a host and a device, such as a printer, scanner, or other peripheral device. It was developed by Xerox in the 1980s and was used in their Xerox Alto computer.

Here are some key features and aspects of the Model Context Protocol:

1. **Device independence**: MCP allows devices to communicate with a host computer in a device-independent manner, meaning that the host can communicate with any device, regardless of its manufacturer or type.
2. **Context-dependent**: The protocol relies on a context-dependent model, where the host and device agree on a common context for communication.
3. **Model-based**: The protocol uses a model-based approach, where the host and device define a model of the data to be exchanged, including the format and structure of the data.
4. **State machine**: MCP uses a state machine model to manage the communication between the host and device, including the transmission a

In [61]:
from google.colab import files
from datasets import load_dataset

# Upload your mcp_sft_dataset.jsonl file
uploaded = files.upload()  # select mcp_sft_dataset.jsonl

dataset = load_dataset("json", data_files="mcp_sft_dataset_v2.jsonl", split="train")
print(f"Total: {len(dataset)}")

# Step 1 — split off test (10%)
train_val = dataset.train_test_split(test_size=0.1, seed=42)

# Step 2 — split off val (10%) from remaining train
train_val_split = train_val["train"].train_test_split(test_size=0.111, seed=42)

train_dataset = train_val_split["train"]   # ~116 examples
val_dataset   = train_val_split["test"]    # ~14  examples
test_dataset  = train_val["test"]          # ~14  examples

print(f"Train : {len(train_dataset)}")
print(f"Val   : {len(val_dataset)}")
print(f"Test  : {len(test_dataset)}")

Saving mcp_sft_dataset_v2.jsonl to mcp_sft_dataset_v2.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Total: 146
Train : 116
Val   : 15
Test  : 15


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [62]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

<a name="Data"></a>
### Data Prep
We now use the `Llama-3.1` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. But we convert it to HuggingFace's normal multiturn format `("role", "content")` instead of `("from", "value")`/ Llama-3 renders multi turn conversations like below:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Hello!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Hey there! How are you?<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm great thanks!<|eot_id|>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3` and more.

In [63]:
original_dataset = dataset

**This is not needed right now but might me needed sometimes**:
We now use `standardize_sharegpt` to convert ShareGPT style datasets into HuggingFace's generic format. This changes the dataset from looking like:
```
{"from": "system", "value": "You are an assistant"}
{"from": "human", "value": "What is 2+2?"}
{"from": "gpt", "value": "It's 4."}
```
to
```
{"role": "system", "content": "You are an assistant"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [70]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    conversations = [
        [
            {"role": "system",    "content": system},
            {"role": "user",      "content": instruction},
            {"role": "assistant", "content": output},
        ]
        for system, instruction, output in zip(
            examples["system"],
            examples["instruction"],
            examples["output"],
        )
    ]
    tokenizer.chat_template = tokenizer.chat_template \
    .replace('{{- "Cutting Knowledge Date: December 2023\n" }}\n', "") \
    .replace('{{- "Today Date: " + date_string + "\n\n" }}\n', "")
    #print(tokenizer.chat_template)

    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in conversations
    ]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset   = val_dataset.map(formatting_prompts_func, batched=True)
test_dataset  = test_dataset.map(formatting_prompts_func, batched=True)

# Verify
print(train_dataset[5]["text"])


Map:   0%|          | 0/116 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert on the Model Context Protocol (MCP), the open standard for connecting AI applications to external tools, data sources, and workflows. Released by Anthropic in November 2024.<|eot_id|><|start_header_id|>user<|end_header_id|>

What happens if a client calls a tool that the server does not support?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

If a client calls a tool that the server does not expose, the server returns a JSON-RPC error with code -32601 (MethodNotFound) or an equivalent application-level error. The client should handle this error gracefully and propagate it to the host, which can inform the LLM and user that the requested tool is not available. Proper capability negotiation and tool listing during initialization should prevent this from happening in well-implemented clients.<|eot_id|>


We look at how the conversations are structured for item 5:

In [68]:
dataset[140]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert on the Model Context Protocol (MCP), the open standard for connecting AI applications to external tools, data sources, and workflows. Released by Anthropic in November 2024.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat happens if a client calls a tool that the server does not support?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nIf a client calls a tool that the server does not expose, the server returns a JSON-RPC error with code -32601 (MethodNotFound) or an equivalent application-level error. The client should handle this error gracefully and propagate it to the host, which can inform the LLM and user that the requested tool is not available. Proper capability negotiation and tool listing during initialization should prevent this from happening in well-implemented clients.<|eot_id|>'

In [ ]:
model

And we see how the chat template transformed these conversations.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

In [69]:
dataset[42]["text"]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert on the Model Context Protocol (MCP), the open standard for connecting AI applications to external tools, data sources, and workflows. Released by Anthropic in November 2024.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDo MCP servers have to implement all three capability types?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nNo, MCP servers do not have to implement all three capability types. A server can expose just tools, just resources, just prompts, or any combination. The server declares which capabilities it supports during the initialization capability negotiation phase, so the client knows exactly what to expect from that server.<|eot_id|>'

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [71]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 3)],  # ← add this
    args = SFTConfig(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 2,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 15,
        save_strategy = "steps",          # ← add this (required for early stopping)
        save_steps = 15,                  # ← must match eval_steps
        load_best_model_at_end = True,    # ← add this (loads best checkpoint on stop)
        metric_for_best_model = "eval_loss",
        greater_is_better = False,        # ← lower eval_loss = better
        optim = "adamw_8bit",
        weight_decay = 0.001,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/116 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=4):   0%|          | 0/15 [00:00<?, ? examples/s]

We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [72]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=4):   0%|          | 0/116 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/116 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/15 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/15 [00:00<?, ? examples/s]

We verify masking is actually done:

In [73]:
trainer.train_dataset[5]

{'input_ids': [128000,
  128000,
  128006,
  9125,
  128007,
  271,
  2675,
  527,
  459,
  6335,
  389,
  279,
  5008,
  9805,
  25590,
  320,
  44,
  7269,
  705,
  279,
  1825,
  5410,
  369,
  21583,
  15592,
  8522,
  311,
  9434,
  7526,
  11,
  828,
  8336,
  11,
  323,
  88568,
  13,
  45894,
  555,
  16989,
  45036,
  304,
  6841,
  220,
  2366,
  19,
  13,
  128009,
  128006,
  882,
  128007,
  271,
  3923,
  8741,
  422,
  264,
  3016,
  6880,
  264,
  5507,
  430,
  279,
  3622,
  1587,
  539,
  1862,
  30,
  128009,
  128006,
  78191,
  128007,
  271,
  2746,
  264,
  3016,
  6880,
  264,
  5507,
  430,
  279,
  3622,
  1587,
  539,
  29241,
  11,
  279,
  3622,
  4780,
  264,
  4823,
  11151,
  4977,
  1493,
  449,
  2082,
  482,
  17470,
  1721,
  320,
  3607,
  10588,
  8,
  477,
  459,
  13890,
  3851,
  11852,
  1493,
  13,
  578,
  3016,
  1288,
  3790,
  420,
  1493,
  79599,
  323,
  58514,
  433,
  311,
  279,
  3552,
  11,
  902,
  649,
  6179,
  279,
  445,
  11

In [47]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are an expert on the Model Context Protocol (MCP), the open standard for connecting AI applications to external tools, data sources, and workflows. Released by Anthropic in November 2024.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhat kinds of external systems can MCP connect AI applications to?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nMCP can connect AI applications to three categories of external systems: data sources such as local files and databases, tools such as search engines and calculators, and workflows such as specialized prompt templates. This covers the full range of things an AI agent might need to access or execute.<|eot_id|>'

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                  Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.<|eot_id|>'

We can see the System and Instruction prompts are successfully masked!

In [74]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
5.662 GB of memory reserved.


In [75]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 116 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
15,2.147700,1.858710
30,1.454400,1.674296
45,1.187300,1.640236
60,1.049700,1.687123


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [76]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

124.6562 seconds used for training.
2.08 minutes used for training.
Peak reserved memory = 5.832 GB.
Peak reserved memory for training = 0.17 GB.
Peak reserved memory % of max memory = 40.047 %.
Peak reserved memory for training % of max memory = 1.167 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [79]:
!pip install rouge-score -q

from rouge_score import rouge_scorer
import json

FastLanguageModel.for_inference(model)

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
results = []

SYSTEM = (
    "You are an expert on the Model Context Protocol (MCP), "
    "the open standard for connecting AI applications to external tools, "
    "data sources, and workflows. Released by Anthropic in November 2024."
)

def run_inference(question):
    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM},
         {"role": "user",   "content": question}],
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

for example in test_dataset:
    predicted = run_inference(example["instruction"])
    scores    = scorer.score(example["output"], predicted)
    results.append({
        "instruction": example["instruction"],
        "expected":    example["output"],
        "predicted":   predicted,
        "rouge1":      round(scores["rouge1"].fmeasure, 3),
        "rouge2":      round(scores["rouge2"].fmeasure, 3),
        "rougeL":      round(scores["rougeL"].fmeasure, 3),
    })

avg_r1 = sum(r["rouge1"] for r in results) / len(results)
avg_r2 = sum(r["rouge2"] for r in results) / len(results)
avg_rL = sum(r["rougeL"] for r in results) / len(results)

print(f"ROUGE-1 : {avg_r1:.3f}")
print(f"ROUGE-2 : {avg_r2:.3f}")
print(f"ROUGE-L : {avg_rL:.3f}  ← primary metric")

print(f"\n{'─'*60}")
for i, r in enumerate(results):
    flag = "✅" if r["rougeL"] >= 0.4 else "⚠️ " if r["rougeL"] >= 0.2 else "❌"
    print(f"\n[{i+1}] {flag} ROUGE-L: {r['rougeL']}")
    print(f"Q  : {r['instruction']}")
    print(f"Exp: {r['expected']}...")
    print(f"Got: {r['predicted']}...")

with open("eval_results.json", "w") as f:
    json.dump({"summary": {"rouge1": avg_r1, "rouge2": avg_r2, "rougeL": avg_rL}, "per_example": results}, f, indent=2)

print(f"\n✅ Saved → eval_results.json")

ROUGE-1 : 0.392
ROUGE-2 : 0.137
ROUGE-L : 0.273  ← primary metric

────────────────────────────────────────────────────────────

[1] ⚠️  ROUGE-L: 0.234
Q  : What does an MCP server expose to clients?
Exp: An MCP server exposes three types of capabilities to clients: tools (functions the LLM can invoke to perform actions), resources (file-like data the client can read for context), and prompts (pre-written templates that help users accomplish specific tasks). Not every server needs to implement all three — a server can expose just tools, just resources, or any combination....
Got: An MCP server exposes resources, tools, and prompts that clients can request and act upon. Resources are data the server provides for clients to access, such as files, databases, or search results. Tools are actions the server can perform on behalf of clients, such as searching files, sending emails, or making API calls. Prompts are templates that clients use to gather information from users, such as file pick

# SFT Evaluation — MCP Fine-tune

## Results

| Question | Status | Issue |
|---|---|---|
| What is MCP? | ✅ Perfect | — |
| Host vs Client vs Server | ⚠️ Partial | Relationship between the three confused |
| Three server capabilities | ❌ Wrong | Said "Tool/Data/Workflow" instead of "Resources/Tools/Prompts" |
| Configure Claude Desktop | ⚠️ Partial | Lost the JSON structure, hallucinated "URL + credentials" |
| What is elicitation? | ✅ Correct | Minor phrasing drift |

## Verdict
SFT worked — base model had zero MCP knowledge, now answers 3/5 correctly.

## Root Cause
Dataset too small (43 examples). Weak spots have only 1–2 covering examples.

## Possible Fix
Add 20–30 targeted examples on the 3 weak spots and retrain.

In [57]:
original_dataset[0]

{'system': 'You are an expert on the Model Context Protocol (MCP), the open standard for connecting AI applications to external tools, data sources, and workflows. Released by Anthropic in November 2024.',
 'instruction': 'What is the Model Context Protocol (MCP)?',
 'input': '',
 'output': 'MCP (Model Context Protocol) is an open-source standard for connecting AI applications to external systems. Using MCP, AI applications like Claude or ChatGPT can connect to data sources such as local files and databases, tools like search engines and calculators, and workflows like specialized prompts. Think of it like a USB-C port for AI — just as USB-C standardizes how you connect devices, MCP standardizes how AI applications connect to external systems.'}

In [56]:
train_dataset = original_dataset["train"]  # 34 examples

ValueError: Column 'train' doesn't exist.

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonacci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

The Fibonacci sequence is a series of numbers where each number is the sum of the two preceding numbers. 

The sequence you provided was: 1, 1, 2, 3, 5, 8, 13

The next number in the sequence would be 21, which is 8 + 13. The sequence continues as: 21, 34, 55, 89, 144, 233.<|eot_id|>

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("llama_lora")  # Local saving
tokenizer.save_pretrained("llama_lora")
# model.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/llama_lora", token = "YOUR_HF_TOKEN") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

The Eiffel Tower, located in the heart of Paris, stands tall among the city's historic and cultural landmarks. This iron structure, standing at an impressive 324 meters high, offers breathtaking views of the City of Light's iconic landscape. The Eiffel Tower was built for the 1889 World's Fair and has since become a symbol of French engineering and culture.<|eot_id|>

You can also use Hugging Face's `AutoPeftModelForCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "llama_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("llama_lora")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("llama_finetune_16bit", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_16bit", tokenizer, save_method = "merged_16bit", token = "YOUR_HF_TOKEN")

# Merge to 4bit
if False: model.save_pretrained_merged("llama_finetune_4bit", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("HF_USERNAME/llama_finetune_4bit", tokenizer, save_method = "merged_4bit", token = "YOUR_HF_TOKEN")

# Just LoRA adapters
if False:
    model.save_pretrained("llama_lora")
    tokenizer.save_pretrained("llama_lora")
if False:
    model.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")
    tokenizer.push_to_hub("HF_USERNAME/llama_lora", token = "YOUR_HF_TOKEN")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("llama_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("llama_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/llama_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/llama_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN", # Get a token at https://huggingface.co/settings/tokens
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>